In [ ]:
# In[1]:

import pandas as pd
import numpy as np
from pathlib import Path

# helper to load and parse timestamp
def load_csv_with_ts(fname, ts_col='timestamp'):
    df = pd.read_csv(fname)
    if ts_col in df.columns:
        df[ts_col] = pd.to_datetime(df[ts_col], unit='s', utc=True)
    return df

# helper to prepare compact unique lists (limit to max_n items)
def compact_unique_list(series, max_n=20):
    uniq = pd.Series(series.dropna().unique())
    # sort strings/numbers for determinism
    try:
        uniq_sorted = sorted(uniq.tolist())
    except Exception:
        uniq_sorted = uniq.tolist()
    truncated = uniq_sorted if len(uniq_sorted) <= max_n else uniq_sorted[:max_n]
    return truncated, len(uniq_sorted)

# Load files (they exist in cwd per task)
metric_df = load_csv_with_ts('metric.csv', 'timestamp')
trace_df  = load_csv_with_ts('trace.csv', 'timestamp')
log_df    = load_csv_with_ts('log.csv', 'timestamp')
error_df  = load_csv_with_ts('error_logs.csv', 'timestamp')

# Summaries
# metric.csv summary
metric_cmdb_list, metric_cmdb_count = compact_unique_list(metric_df['cmdb_id'])
metric_kpi_list, metric_kpi_count = compact_unique_list(metric_df['kpi_name'])
metric_earliest = metric_df['timestamp'].min() if 'timestamp' in metric_df.columns and not metric_df['timestamp'].isna().all() else pd.NaT
metric_latest   = metric_df['timestamp'].max() if 'timestamp' in metric_df.columns and not metric_df['timestamp'].isna().all() else pd.NaT
metric_summary = pd.DataFrame([{
    'file': 'metric.csv',
    'total_rows': len(metric_df),
    'unique_cmdb_ids_shown': metric_cmdb_list,
    'unique_cmdb_ids_count': metric_cmdb_count,
    'unique_kpi_names_shown': metric_kpi_list,
    'unique_kpi_names_count': metric_kpi_count,
    'earliest_utc': metric_earliest,
    'latest_utc': metric_latest
}])

# trace.csv summary
trace_cmdb_list, trace_cmdb_count = compact_unique_list(trace_df['cmdb_id'])
trace_name_list, trace_name_count = compact_unique_list(trace_df['trace_name'])
trace_earliest = trace_df['timestamp'].min() if 'timestamp' in trace_df.columns and not trace_df['timestamp'].isna().all() else pd.NaT
trace_latest   = trace_df['timestamp'].max() if 'timestamp' in trace_df.columns and not trace_df['timestamp'].isna().all() else pd.NaT
trace_summary = pd.DataFrame([{
    'file': 'trace.csv',
    'total_rows': len(trace_df),
    'unique_cmdb_ids_shown': trace_cmdb_list,
    'unique_cmdb_ids_count': trace_cmdb_count,
    'unique_trace_names_shown': trace_name_list,
    'unique_trace_names_count': trace_name_count,
    'earliest_utc': trace_earliest,
    'latest_utc': trace_latest
}])

# log.csv summary
log_cmdb_list, log_cmdb_count = compact_unique_list(log_df['cmdb_id'])
log_name_list, log_name_count = compact_unique_list(log_df['log_name'])
log_earliest = log_df['timestamp'].min() if 'timestamp' in log_df.columns and not log_df['timestamp'].isna().all() else pd.NaT
log_latest   = log_df['timestamp'].max() if 'timestamp' in log_df.columns and not log_df['timestamp'].isna().all() else pd.NaT
log_summary = pd.DataFrame([{
    'file': 'log.csv',
    'total_rows': len(log_df),
    'unique_cmdb_ids_shown': log_cmdb_list,
    'unique_cmdb_ids_count': log_cmdb_count,
    'unique_log_names_shown': log_name_list,
    'unique_log_names_count': log_name_count,
    'earliest_utc': log_earliest,
    'latest_utc': log_latest
}])

# error_logs.csv summary: sample up to 20 distinct message snippets or markers
if 'message' in error_df.columns:
    # get up to 20 distinct messages (truncate each message to 200 chars to keep compact)
    uniq_messages = pd.Series(error_df['message'].dropna().astype(str).unique())
    uniq_count = len(uniq_messages)
    sampled_msgs = uniq_messages.tolist()[:20]
    sampled_msgs_trunc = [m if len(m) <= 200 else m[:200] + '...[truncated]' for m in sampled_msgs]
else:
    sampled_msgs_trunc = []
    uniq_count = 0
error_earliest = error_df['timestamp'].min() if 'timestamp' in error_df.columns and not error_df['timestamp'].isna().all() else pd.NaT
error_latest   = error_df['timestamp'].max() if 'timestamp' in error_df.columns and not error_df['timestamp'].isna().all() else pd.NaT
error_summary = pd.DataFrame([{
    'file': 'error_logs.csv',
    'total_rows': len(error_df),
    'unique_cmdb_ids_shown': compact_unique_list(error_df['cmdb_id'])[0],
    'unique_cmdb_ids_count': compact_unique_list(error_df['cmdb_id'])[1],
    'sample_messages_shown_up_to_20': sampled_msgs_trunc,
    'unique_message_count': uniq_count,
    'earliest_utc': error_earliest,
    'latest_utc': error_latest
}])

# For metric.csv compute global P95 for each (cmdb_id, kpi_name) using full series
# Ensure value column exists and is numeric
metric_p95 = None
if {'cmdb_id', 'kpi_name', 'value'}.issubset(metric_df.columns):
    # drop NA values in value
    metric_p95 = (metric_df.dropna(subset=['value'])
                  .groupby(['cmdb_id', 'kpi_name'], dropna=False)['value']
                  .quantile(0.95)
                  .reset_index()
                  .rename(columns={'value': 'p95'}))
    # sort by p95 desc and take top 20
    metric_p95_top20 = metric_p95.sort_values('p95', ascending=False).head(20).reset_index(drop=True)
else:
    metric_p95_top20 = pd.DataFrame(columns=['cmdb_id', 'kpi_name', 'p95'])

# Final compact outputs
metric_summary, trace_summary, log_summary, error_summary, metric_p95_top20

```
Out[1]:
```
Summary (plain English):

- metric.csv
  - Total rows: 9,175
  - Unique cmdb_id count: 68 (sample shown includes ts-admin-basic-info-service, ts-admin-order-service, etc.)
  - Unique KPI names: 8 (examples: cpu, diskio, error, latency-50, latency-90, ...)
  - Earliest timestamp: 2024-01-23 16:06:00 UTC
  - Latest timestamp: 2024-01-23 16:30:00 UTC

- trace.csv
  - Total rows: 18,656
  - Unique cmdb_id count: 28 (sample includes root, ts-admin-basic-info-service, ts-admin-...)
  - Unique trace names: 220 (many trace features)
  - Earliest timestamp: 2024-01-23 16:06:00 UTC
  - Latest timestamp: 2024-01-23 16:30:00 UTC

- log.csv
  - Total rows: 2,194
  - Unique cmdb_id count: 45 (sample includes ts-admin-basic-info-service, ts-admin-travel-...)
  - Unique log names: 2 (log.error_count, log.row_count)
  - Earliest timestamp: 2024-01-23 16:06:00 UTC
  - Latest timestamp: 2024-01-23 16:30:00 UTC

- error_logs.csv
  - Total rows: 0
  - Unique cmdb_id count: 0
  - No messages or timestamps present

- Global metric P95 (computed per (cmdb_id, kpi_name); top 20 by p95)
  - All top-20 entries are the memory ("mem") KPI.
  - Highest P95: ts-auth-service (mem) = 6.2513e+08
  - Next highest: ts-food-mysql (mem) ≈ 5.9989e+08 and ts-voucher-mysql (mem) ≈ 5.9986e+08
  - The 20th entry (lowest in this top-20) is ts-consign-service (mem) ≈ 2.4892e+08

Interpretation note: the dominant high P95 values are memory usage across services; ts-auth-service shows the single highest P95 in the available metric series.

The original code execution output of IPython Kernel is also provided below for reference:

(         file  total_rows                              unique_cmdb_ids_shown  unique_cmdb_ids_count                             unique_kpi_names_shown  unique_kpi_names_count              earliest_utc                latest_utc
0  metric.csv        9175  [ts-admin-basic-info-service, ts-admin-order-s...                     68  [cpu, diskio, error, latency-50, latency-90, m...                       8 2024-01-23 16:06:00+00:00 2024-01-23 16:30:00+00:00,         file  total_rows                              unique_cmdb_ids_shown  unique_cmdb_ids_count                           unique_trace_names_shown  unique_trace_names_count              earliest_utc                latest_utc
0  trace.csv       18656  [root, ts-admin-basic-info-service, ts-admin-t...                     28  [trace.from_root.duration_mean, trace.from_roo...                       220 2024-01-23 16:06:00+00:00 2024-01-23 16:30:00+00:00,       file  total_rows                              unique_cmdb_ids_shown  unique_cmdb_ids_count            unique_log_names_shown  unique_log_names_count              earliest_utc                latest_utc
0  log.csv        2194  [ts-admin-basic-info-service, ts-admin-travel-...                     45  [log.error_count, log.row_count]                       2 2024-01-23 16:06:00+00:00 2024-01-23 16:30:00+00:00,              file  total_rows unique_cmdb_ids_shown  unique_cmdb_ids_count sample_messages_shown_up_to_20  unique_message_count earliest_utc latest_utc
0  error_logs.csv           0                    []                      0                             []                     0          NaT        NaT,                       cmdb_id kpi_name           p95
0             ts-auth-service      mem  6.251300e+08
1               ts-food-mysql      mem  5.998879e+08
2            ts-voucher-mysql      mem  5.998633e+08
3          ts-travel2-service      mem  2.833184e+08
4      ts-order-other-service      mem  2.755242e+08
5           ts-travel-service      mem  2.750427e+08
6            ts-order-service      mem  2.724577e+08
7          ts-station-service      mem  2.678671e+08
8             ts-food-service      mem  2.629559e+08
9            ts-price-service      mem  2.596945e+08
10           ts-route-service      mem  2.586344e+08
11        ts-food-map-service      mem  2.582831e+08
12          ts-config-service      mem  2.549089e+08
13           ts-train-service      mem  2.530616e+08
14        ts-contacts-service      mem  2.517208e+08
15  ts-inside-payment-service      mem  2.514459e+08
16       ts-assurance-service      mem  2.507061e+08
17           ts-basic-service      mem  2.498123e+08
18        ts-security-service      mem  2.497557e+08
19         ts-consign-service      mem  2.489238e+08)```
```

In [ ]:
# In[2]:

import pandas as pd
from pandas import Timestamp

# reuse metric_df loaded previously in the session
# define candidates and incident window (UTC)
candidates = ["ts-auth-service","ts-order-service","ts-route-service","ts-train-service","ts-travel-service"]
start = pd.to_datetime('2024-01-23 16:03:20', utc=True)
end   = pd.to_datetime('2024-01-23 16:33:20', utc=True)

# ensure metric_df has expected columns and timestamp is datetime UTC
assert {'timestamp','cmdb_id','kpi_name','value'}.issubset(metric_df.columns)

# 1) compute global P95 per (cmdb_id, kpi_name) using full series (ONLY for candidate cmdb_ids)
metric_candidates_full = metric_df[metric_df['cmdb_id'].isin(candidates)].copy()
# drop NA in value for threshold calculation
metric_p95 = (metric_candidates_full
              .dropna(subset=['value'])
              .groupby(['cmdb_id','kpi_name'], dropna=False)['value']
              .quantile(0.95)
              .reset_index()
              .rename(columns={'value':'global_p95'}))

# 2) filter metric rows to the incident window and candidates
metric_window = metric_df[
    (metric_df['cmdb_id'].isin(candidates)) &
    (metric_df['timestamp'] >= start) &
    (metric_df['timestamp'] <= end)
].copy()

# If no rows in window, return empty df with requested columns
if metric_window.shape[0] == 0:
    result_df = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','global_p95','count_in_window','anomaly_count',
        'earliest_anomaly_timestamp','max_value_in_window','sample_anomalies'
    ])
else:
    # join global_p95 into window rows to mark anomalies (global thresholds computed before filtering)
    metric_win_with_p95 = metric_window.merge(metric_p95, on=['cmdb_id','kpi_name'], how='left')

    # compute aggregates per pair
    grp = metric_win_with_p95.groupby(['cmdb_id','kpi_name'], sort=False)

    rows = []
    for (cmdb, kpi), sub in grp:
        count_in_window = int(len(sub))
        # determine global_p95 (may be NaN if p95 couldn't be computed because full series had no values)
        gp95 = sub['global_p95'].iloc[0] if 'global_p95' in sub.columns else float('nan')
        # anomalies: value >= global_p95 (only if global_p95 is not NaN)
        if pd.notna(gp95):
            anomalies = sub[sub['value'] >= gp95].sort_values('timestamp', ascending=True)
        else:
            anomalies = sub.iloc[0:0]  # empty
        anomaly_count = int(len(anomalies))
        earliest_anom_ts = anomalies['timestamp'].min() if anomaly_count>0 else pd.NaT
        max_value = sub['value'].max() if count_in_window>0 else None
        # sample up to 5 anomaly rows as strings "YYYY-MM-DDTHH:MM:SS+00:00 | value"
        sample_anoms = []
        if anomaly_count>0:
            samp = anomalies.head(5)
            # keep compact textual representation
            sample_anoms = [f"{ts.isoformat()} | {val}" for ts, val in zip(samp['timestamp'], samp['value'])]
        rows.append({
            'cmdb_id': cmdb,
            'kpi_name': kpi,
            'global_p95': gp95,
            'count_in_window': count_in_window,
            'anomaly_count': anomaly_count,
            'earliest_anomaly_timestamp': earliest_anom_ts,
            'max_value_in_window': max_value,
            'sample_anomalies': sample_anoms
        })

    result_df = pd.DataFrame(rows)

    # only include pairs with count_in_window > 0 (should already be true) and sort by anomaly_count desc then cmdb_id
    result_df = result_df[result_df['count_in_window']>0].sort_values(['anomaly_count','cmdb_id'], ascending=[False, True]).reset_index(drop=True)

    # limit to top 20 rows
    result_df = result_df.head(20)

# final compact output
result_df

```
Out[2]:
```
Summary (plain English):

- Scope: I examined metrics for candidates ["ts-auth-service","ts-order-service","ts-route-service","ts-train-service","ts-travel-service"] inside the incident window 2024-01-23 16:03:20 → 2024-01-23 16:33:20 UTC. Only (cmdb_id,kpi_name) pairs with data in the window are reported.

Key findings (top observations):
1. Error KPI (log-like metric)
   - ts-order-service (kpi: error): global P95 = 0.0, count in window = 25, anomaly_count = 25, earliest anomaly = 2024-01-23 16:06:00 UTC, max = 0.0.
   - ts-travel-service (kpi: error): global P95 = 0.0, count in window = 25, anomaly_count = 25, earliest anomaly = 2024-01-23 16:06:00 UTC, max = 0.0.
   - Note: global P95 = 0 means every zero-valued sample meets ">= p95", so these anomaly counts are an artifact of the threshold and should be treated with caution.

2. ts-auth-service (multiple KPIs): each KPI had 25 samples in-window and 2 anomaly samples (value >= global P95):
   - cpu: global_p95 ≈ 39.662, anomaly_count = 2, earliest anomaly = 2024-01-23 16:20:00 UTC, max_in_window ≈ 39.7587
   - diskio: global_p95 ≈ 24,562,850, anomaly_count = 2, earliest anomaly = 2024-01-23 16:20:00 UTC, max_in_window ≈ 24,583,000
   - latency-50: global_p95 ≈ 1.3790, anomaly_count = 2, earliest anomaly = 2024-01-23 16:24:00 UTC, max_in_window ≈ 1.4294
   - latency-90: global_p95 ≈ 2.33998, anomaly_count = 2, earliest anomaly = 2024-01-23 16:25:00 UTC, max_in_window ≈ 2.3953
   - mem: global_p95 ≈ 625,130,000, anomaly_count = 2, earliest anomaly = 2024-01-23 16:22:00 UTC, max_in_window ≈ 681,281,263
   - socket: global_p95 ≈ 19.6767, anomaly_count = 2, earliest anomaly = 2024-01-23 16:27:00 UTC, max_in_window ≈ 20.4167
   - workload: global_p95 ≈ 2.8141, anomaly_count = 2, earliest anomaly = 2024-01-23 16:09:00 UTC, max_in_window ≈ 2.8288

3. ts-order-service (besides error) had multiple KPIs with 2 anomalies each:
   - cpu: global_p95 ≈ 2.37145, anomaly_count = 2, earliest anomaly = 2024-01-23 16:10:00 UTC, max ≈ 2.4384
   - diskio: global_p95 ≈ 60,197.31, anomaly_count = 2, earliest anomaly = 2024-01-23 16:12:00 UTC, max ≈ 60,719.40
   - latency-50: global_p95 ≈ 0.0096555, anomaly_count = 2, earliest anomaly = 2024-01-23 16:11:00 UTC, max ≈ 0.0099456
   - latency-90: global_p95 ≈ 0.0238327, anomaly_count = 2, earliest anomaly = 2024-01-23 16:23:00 UTC, max ≈ 0.0377228
   - mem: global_p95 ≈ 272,457,700, anomaly_count = 2, earliest anomaly = 2024-01-23 16:29:00 UTC, max ≈ 272,708,963
   - socket: global_p95 ≈ 12.85, anomaly_count = 2, earliest anomaly = 2024-01-23 16:16:00 UTC, max ≈ 13.8667
   - workload: global_p95 ≈ 6.9087, anomaly_count = 2, earliest anomaly = 2024-01-23 16:14:00 UTC, max ≈ 8.3267

4. ts-route-service (several KPIs with 2 anomalies each):
   - cpu: global_p95 ≈ 4.6714, anomaly_count = 2, earliest anomaly = 2024-01-23 16:12:00 UTC, max ≈ 4.7260
   - diskio: global_p95 ≈ 1,363,627, anomaly_count = 2, earliest anomaly = 2024-01-23 16:22:00 UTC, max ≈ 1,406,358
   - latency-50: global_p95 ≈ 0.0034189, anomaly_count = 2, earliest anomaly = 2024-01-23 16:26:00 UTC, max ≈ 0.0034397
   - latency-90: global_p95 ≈ 0.0073824, anomaly_count = 2, earliest anomaly = 2024-01-23 16:26:00 UTC, max ≈ 0.0075626

5. ts-train-service and ts-travel-service: in the returned results only ts-travel-service appeared for the "error" KPI (see #1). ts-train-service had no top entries shown among the top-20 rows returned.

Practical interpretation:
- The most striking counts are the "error" KPI rows for ts-order-service and ts-travel-service (25 anomalies each). However, because the global P95 for error is 0, every zero-valued sample meets the threshold — this is likely not a meaningful anomaly signal and needs manual inspection of what "error" value encodes.
- Among meaningful resource/latency KPIs, ts-auth-service shows several KP95 breaches (cpu, mem, diskio, latencies) with small numbers of anomalous points (2 each) and notable peaks (e.g., mem peak ≈ 681M for ts-auth-service).
- ts-order-service and ts-route-service also show isolated peaks (2 anomaly points) across CPU, mem, disk I/O and latency KPIs.

Recommended next steps:
- Inspect the raw "error" metric semantics for ts-order-service and ts-travel-service to determine why P95 = 0 and whether zeros should be treated as anomalies.
- For ts-auth-service (and ts-order/service, ts-route-service), investigate the specific timestamps of the anomalous samples (listed in the sample_anomalies) for correlation with traces/logs around those times (notably ~16:20–16:29 UTC) to confirm root cause.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id    kpi_name    global_p95  count_in_window  anomaly_count earliest_anomaly_timestamp  max_value_in_window                                   sample_anomalies
0    ts-order-service       error  0.000000e+00               25             25  2024-01-23 16:06:00+00:00         0.000000e+00  [2024-01-23T16:06:00+00:00 | 0.0, 2024-01-23T1...
1   ts-travel-service       error  0.000000e+00               25             25  2024-01-23 16:06:00+00:00         0.000000e+00  [2024-01-23T16:06:00+00:00 | 0.0, 2024-01-23T1...
2     ts-auth-service         cpu  3.966213e+01               25              2  2024-01-23 16:20:00+00:00         3.975867e+01  [2024-01-23T16:20:00+00:00 | 39.75867193931754...
3     ts-auth-service      diskio  2.456285e+07               25              2  2024-01-23 16:20:00+00:00         2.535829e+07  [2024-01-23T16:20:00+00:00 | 24590019.63882929...
4     ts-auth-service  latency-50  1.379032e+00               25              2  2024-01-23 16:24:00+00:00         1.429371e+00  [2024-01-23T16:24:00+00:00 | 1.382195143623685...
5     ts-auth-service  latency-90  2.339984e+00               25              2  2024-01-23 16:25:00+00:00         2.395314e+00  [2024-01-23T16:25:00+00:00 | 2.395314148224305...
6     ts-auth-service         mem  6.251300e+08               25              2  2024-01-23 16:22:00+00:00         6.812813e+08  [2024-01-23T16:22:00+00:00 | 681281262.9333333...
7     ts-auth-service      socket  1.967667e+01               25              2  2024-01-23 16:27:00+00:00         2.041667e+01  [2024-01-23T16:27:00+00:00 | 20.41666666666666...
8     ts-auth-service    workload  2.814053e+00               25              2  2024-01-23 16:09:00+00:00         2.828783e+00  [2024-01-23T16:09:00+00:00 | 2.828783333333333...
9    ts-order-service         cpu  2.371450e+00               25              2  2024-01-23 16:10:00+00:00         2.438436e+00  [2024-01-23T16:10:00+00:00 | 2.377546302163425...
10   ts-order-service      diskio  6.019731e+04               25              2  2024-01-23 16:12:00+00:00         6.071940e+04  [2024-01-23T16:12:00+00:00 | 60719.40262130962...
11   ts-order-service  latency-50  9.655494e-03               25              2  2024-01-23 16:11:00+00:00         9.945563e-03  [2024-01-23T16:11:00+00:00 | 0.009705889880808...
12   ts-order-service  latency-90  2.383273e-02               25              2  2024-01-23 16:23:00+00:00         3.772284e-02  [2024-01-23T16:23:00+00:00 | 0.037722839008222...
13   ts-order-service         mem  2.724577e+08               25              2  2024-01-23 16:29:00+00:00         2.727089e+08  [2024-01-23T16:29:00+00:00 | 272561629.8666667...
14   ts-order-service      socket  1.285000e+01               25              2  2024-01-23 16:16:00+00:00         1.386667e+01  [2024-01-23T16:16:00+00:00 | 12.86666666666666...
15   ts-order-service    workload  6.908723e+00               25              2  2024-01-23 16:14:00+00:00         8.326700e+00  [2024-01-23T16:14:00+00:00 | 8.326699999999999...
16   ts-route-service         cpu  4.671403e+00               25              2  2024-01-23 16:12:00+00:00         4.725991e+00  [2024-01-23T16:12:00+00:00 | 4.725991146129774...
17   ts-route-service      diskio  1.363627e+06               25              2  2024-01-23 16:22:00+00:00         1.406358e+06  [2024-01-23T16:22:00+00:00 | 1378662.260985191...
18   ts-route-service  latency-50  3.418927e-03               25              2  2024-01-23 16:26:00+00:00         3.439668e-03  [2024-01-23T16:26:00+00:00 | 0.003439668483093...
19   ts-route-service  latency-90  7.382447e-03               25              2  2024-01-23 16:26:00+00:00         7.562625e-03  [2024-01-23T16:26:00+00:00 | 0.007562625122558...```
```

In [ ]:
# In[3]:

import pandas as pd

# reuse trace_df and start/end from previous session if available; otherwise load and parse
try:
    trace_df
except NameError:
    trace_df = pd.read_csv('trace.csv')
    trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)

# ensure timestamp is datetime UTC
if trace_df['timestamp'].dtype == object:
    trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)

# candidates and incident window (UTC)
candidates = ["ts-auth-service","ts-order-service","ts-route-service","ts-travel-service"]
start = pd.to_datetime('2024-01-23 16:03:20', utc=True)
end   = pd.to_datetime('2024-01-23 16:33:20', utc=True)

# 1) compute global P95 per (cmdb_id, trace_name) using full series for candidate cmdb_ids
trace_candidates_full = trace_df[trace_df['cmdb_id'].isin(candidates)].copy()
trace_p95 = (trace_candidates_full
             .dropna(subset=['value'])
             .groupby(['cmdb_id','trace_name'], dropna=False)['value']
             .quantile(0.95)
             .reset_index()
             .rename(columns={'value':'global_p95'}))

# 2) filter traces to incident window and candidate cmdb_ids
trace_window = trace_df[
    (trace_df['cmdb_id'].isin(candidates)) &
    (trace_df['timestamp'] >= start) &
    (trace_df['timestamp'] <= end)
].copy()

# prepare result rows
if trace_window.shape[0] == 0:
    trace_result_df = pd.DataFrame(columns=[
        'cmdb_id','trace_name','global_p95','count_in_window','anomaly_count',
        'earliest_anomaly_timestamp','max_value_in_window','sample_anomalies'
    ])
else:
    # join global_p95 into window rows (threshold computed globally before filtering)
    win_with_p95 = trace_window.merge(trace_p95, on=['cmdb_id','trace_name'], how='left')

    rows = []
    grp = win_with_p95.groupby(['cmdb_id','trace_name'], sort=False)
    for (cmdb, tname), sub in grp:
        count_in_window = int(len(sub))
        gp95 = sub['global_p95'].iloc[0] if 'global_p95' in sub.columns else float('nan')
        if pd.notna(gp95):
            anomalies = sub[sub['value'] >= gp95].sort_values('timestamp', ascending=True)
        else:
            anomalies = sub.iloc[0:0]
        anomaly_count = int(len(anomalies))
        earliest_anom_ts = anomalies['timestamp'].min() if anomaly_count>0 else pd.NaT
        max_value = sub['value'].max() if count_in_window>0 else None
        sample_anoms = []
        if anomaly_count>0:
            samp = anomalies.head(5)
            sample_anoms = [f"{ts.isoformat()} | {val}" for ts, val in zip(samp['timestamp'], samp['value'])]
        rows.append({
            'cmdb_id': cmdb,
            'trace_name': tname,
            'global_p95': gp95,
            'count_in_window': count_in_window,
            'anomaly_count': anomaly_count,
            'earliest_anomaly_timestamp': earliest_anom_ts,
            'max_value_in_window': max_value,
            'sample_anomalies': sample_anoms
        })

    trace_result_df = pd.DataFrame(rows)
    # only include pairs with count_in_window > 0 (they should all have >0 by construction)
    trace_result_df = trace_result_df[trace_result_df['count_in_window']>0]
    # sort by anomaly_count desc then cmdb_id asc, then trace_name
    trace_result_df = trace_result_df.sort_values(['anomaly_count','cmdb_id','trace_name'], ascending=[False, True, True]).reset_index(drop=True)
    # limit to top 20
    trace_result_df = trace_result_df.head(20)

# final compact output
trace_result_df

```
Out[3]:
```
Summary (plain English):

- Scope: I examined trace telemetry for cmdb_ids ["ts-auth-service","ts-order-service","ts-route-service","ts-travel-service"] in the incident window 2024-01-23 16:03:20 → 16:33:20 UTC. Only trace_name pairs with data in-window are reported.

Key result:
- All reported trace features are error_rate metrics (various directions like trace.from_... or trace.to_...).
- For every reported (cmdb_id, trace_name) the global P95 = 0.0, count_in_window = 25, anomaly_count = 25, earliest anomaly = 2024-01-23 16:06:00 UTC, and max_value_in_window = 0.0.
- Example services and trace_names include:
  - ts-auth-service: trace.from_root.error_rate, trace.from_ts-auth-service.error_rate, trace.to_ts-auth-service.error_rate (and similar)
  - ts-order-service: trace.from_root.error_rate, trace.from_ts-order-service.error_rate, trace.to_ts-order-service.error_rate, etc.
  - ts-route-service: trace.from_ts-route-service.error_rate, trace.to_ts-route-service.error_rate, etc.
  - ts-travel-service: trace.from_root.error_rate, trace.from_ts-travel-service.error_rate, etc.

Interpretation:
- The anomaly counts are an artifact: global P95 = 0.0 and all values in-window are 0.0, so every sample meets value >= P95. This does not indicate meaningful elevated error rates.
- No non-zero error_rate spikes were observed in the window for these trace error_rate metrics.

Recommendation:
- Treat these "anomalies" as threshold artifacts. Verify the semantics of the error_rate trace fields (why P95 = 0 and whether zero means "no errors" or missing/normalized data).
- Investigate other trace or metric KPIs (durations, latencies, resource metrics) for real signals — e.g., earlier metric analysis showed ts-auth-service had small numbers of breaches in cpu/mem/diskio/latency worth following up.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                     trace_name  global_p95  count_in_window  anomaly_count earliest_anomaly_timestamp  max_value_in_window                                   sample_anomalies
0     ts-auth-service                     trace.from_root.error_rate         0.0               25             25  2024-01-23 16:06:00+00:00                  0.0  [2024-01-23T16:06:00+00:00 | 0.0, 2024-01-23T1...
1     ts-auth-service          trace.from_ts-auth-service.error_rate         0.0               25             25  2024-01-23 16:06:00+00:00                  0.0  [2024-01-23T16:06:00+00:00 | 0.0, 2024-01-23T1...
2     ts-auth-service            trace.to_ts-auth-service.error_rate         0.0               25             25  2024-01-23 16:06:00+00:00                  0.0  [2024-01-23T16:06:00+00:00 | 0.0, 2024-01-23T1...
3    ts-order-service                     trace.from_root.error_rate         0.0               25             25  2024-01-23 16:06:00+00:00                  0.0  [2024-01-23T16:06:00+00:00 | 0.0, 2024-01-23T1...
4    ts-order-service         trace.from_ts-order-service.error_rate         0.0               25             25  2024-01-23 16:06:00+00:00                  0.0  [2024-01-23T16:06:00+00:00 | 0.0, 2024-01-23T1...
5    ts-order-service          trace.from_ts-seat-service.error_rate         0.0               25             25  2024-01-23 16:06:00+00:00                  0.0  [2024-01-23T16:06:00+00:00 | 0.0, 2024-01-23T1...
6    ts-order-service      trace.from_ts-security-service.error_rate         0.0               25             25  2024-01-23 16:06:00+00:00                  0.0  [2024-01-23T16:06:00+00:00 | 0.0, 2024-01-23T1...
7    ts-order-service        trace.from_ts-travel-service.error_rate         0.0               25             25  2024-01-23 16:06:00+00:00                  0.0  [2024-01-23T16:06:00+00:00 | 0.0, 2024-01-23T1...
8    ts-order-service           trace.to_ts-order-service.error_rate         0.0               25             25  2024-01-23 16:06:00+00:00                  0.0  [2024-01-23T16:06:00+00:00 | 0.0, 2024-01-23T1...
9    ts-order-service         trace.to_ts-station-service.error_rate         0.0               25             25  2024-01-23 16:06:00+00:00                  0.0  [2024-01-23T16:06:00+00:00 | 0.0, 2024-01-23T1...
10   ts-route-service         trace.from_ts-basic-service.error_rate         0.0               25             25  2024-01-23 16:06:00+00:00                  0.0  [2024-01-23T16:06:00+00:00 | 0.0, 2024-01-23T1...
11   ts-route-service         trace.from_ts-route-service.error_rate         0.0               25             25  2024-01-23 16:06:00+00:00                  0.0  [2024-01-23T16:06:00+00:00 | 0.0, 2024-01-23T1...
12   ts-route-service        trace.from_ts-travel-service.error_rate         0.0               25             25  2024-01-23 16:06:00+00:00                  0.0  [2024-01-23T16:06:00+00:00 | 0.0, 2024-01-23T1...
13   ts-route-service       trace.from_ts-travel2-service.error_rate         0.0               25             25  2024-01-23 16:06:00+00:00                  0.0  [2024-01-23T16:06:00+00:00 | 0.0, 2024-01-23T1...
14   ts-route-service           trace.to_ts-route-service.error_rate         0.0               25             25  2024-01-23 16:06:00+00:00                  0.0  [2024-01-23T16:06:00+00:00 | 0.0, 2024-01-23T1...
15  ts-travel-service                     trace.from_root.error_rate         0.0               25             25  2024-01-23 16:06:00+00:00                  0.0  [2024-01-23T16:06:00+00:00 | 0.0, 2024-01-23T1...
16  ts-travel-service  trace.from_ts-admin-travel-service.error_rate         0.0               25             25  2024-01-23 16:06:00+00:00                  0.0  [2024-01-23T16:06:00+00:00 | 0.0, 2024-01-23T1...
17  ts-travel-service          trace.from_ts-food-service.error_rate         0.0               25             25  2024-01-23 16:06:00+00:00                  0.0  [2024-01-23T16:06:00+00:00 | 0.0, 2024-01-23T1...
18  ts-travel-service          trace.from_ts-seat-service.error_rate         0.0               25             25  2024-01-23 16:06:00+00:00                  0.0  [2024-01-23T16:06:00+00:00 | 0.0, 2024-01-23T1...
19  ts-travel-service        trace.from_ts-travel-service.error_rate         0.0               25             25  2024-01-23 16:06:00+00:00                  0.0  [2024-01-23T16:06:00+00:00 | 0.0, 2024-01-23T1...```
```

In [ ]:
# In[4]:

import pandas as pd

# reuse variables if present; otherwise load files
try:
    log_df
except NameError:
    log_df = pd.read_csv('log.csv')
    log_df['timestamp'] = pd.to_datetime(log_df['timestamp'], unit='s', utc=True)

try:
    error_df
except NameError:
    error_df = pd.read_csv('error_logs.csv')
    error_df['timestamp'] = pd.to_datetime(error_df['timestamp'], unit='s', utc=True)

# ensure timestamp columns are datetime UTC
if log_df['timestamp'].dtype == object or not pd.api.types.is_datetime64_any_dtype(log_df['timestamp']):
    log_df['timestamp'] = pd.to_datetime(log_df['timestamp'], unit='s', utc=True)
if 'timestamp' in error_df.columns and (error_df['timestamp'].dtype == object or not pd.api.types.is_datetime64_any_dtype(error_df['timestamp'])):
    error_df['timestamp'] = pd.to_datetime(error_df['timestamp'], unit='s', utc=True)

# parameters
candidates = ["ts-auth-service","ts-order-service","ts-route-service","ts-travel-service"]
start = pd.to_datetime('2024-01-23 16:03:20', utc=True)
end   = pd.to_datetime('2024-01-23 16:33:20', utc=True)

# 1) compute global P95 per (cmdb_id, log_name) using the full series but limited to candidates
log_candidates_full = log_df[log_df['cmdb_id'].isin(candidates)].copy()

if {'cmdb_id','log_name','value'}.issubset(log_candidates_full.columns) and log_candidates_full.dropna(subset=['value']).shape[0] > 0:
    log_p95 = (log_candidates_full
               .dropna(subset=['value'])
               .groupby(['cmdb_id','log_name'], dropna=False)['value']
               .quantile(0.95)
               .reset_index()
               .rename(columns={'value':'global_p95'}))
else:
    log_p95 = pd.DataFrame(columns=['cmdb_id','log_name','global_p95'])

# 2) filter log rows to incident window and candidates
log_window = log_df[
    (log_df['cmdb_id'].isin(candidates)) &
    (log_df['timestamp'] >= start) &
    (log_df['timestamp'] <= end)
].copy()

# 3) produce compact summary per (cmdb_id, log_name) with required fields, only if count_in_window>0
rows = []
if log_window.shape[0] > 0:
    win_with_p95 = log_window.merge(log_p95, on=['cmdb_id','log_name'], how='left')
    grp = win_with_p95.groupby(['cmdb_id','log_name'], sort=False)
    for (cmdb, lname), sub in grp:
        count_in_window = int(len(sub))
        gp95 = sub['global_p95'].iloc[0] if 'global_p95' in sub.columns else float('nan')
        # anomalies: value >= global_p95 (if gp95 is NaN then anomalies empty)
        if pd.notna(gp95):
            anomalies = sub[sub['value'] >= gp95].sort_values('timestamp', ascending=True)
        else:
            anomalies = sub.iloc[0:0]
        anomaly_count = int(len(anomalies))
        earliest_anom_ts = anomalies['timestamp'].min() if anomaly_count>0 else pd.NaT
        max_value = sub['value'].max() if count_in_window>0 else None
        # up to 10 sample rows from the window (timestamp | value) ordered by timestamp ascending
        samp = sub.sort_values('timestamp', ascending=True).head(10)
        sample_rows = [f"{ts.isoformat()} | {val}" for ts, val in zip(samp['timestamp'], samp['value'])]
        rows.append({
            'cmdb_id': cmdb,
            'log_name': lname,
            'global_p95': gp95,
            'count_in_window': count_in_window,
            'anomaly_count': anomaly_count,
            'earliest_anomaly_timestamp': earliest_anom_ts,
            'max_value_in_window': max_value,
            'sample_window_rows_up_to_10': sample_rows
        })

log_result_df = pd.DataFrame(rows)

# only include pairs with count_in_window > 0 (already satisfied) and sort by anomaly_count desc then cmdb_id
if not log_result_df.empty:
    log_result_df = log_result_df.sort_values(['anomaly_count','cmdb_id'], ascending=[False, True]).reset_index(drop=True)
    log_result_df = log_result_df.head(20)

# For error_logs.csv: filter to incident window and candidates and return up to 20 rows (timestamp UTC, cmdb_id, message)
if 'timestamp' in error_df.columns:
    error_window = error_df[
        (error_df['cmdb_id'].isin(candidates)) &
        (error_df['timestamp'] >= start) &
        (error_df['timestamp'] <= end)
    ].copy()
    # keep only required columns and limit to 20 rows ordered by timestamp asc
    if not error_window.empty:
        # ensure message column exists
        if 'message' not in error_window.columns:
            error_window['message'] = ''
        error_window_display = error_window.sort_values('timestamp', ascending=True)[['timestamp','cmdb_id','message']].head(20).reset_index(drop=True)
    else:
        error_window_display = error_window.head(0)[['timestamp','cmdb_id','message']] if 'message' in error_window.columns else pd.DataFrame(columns=['timestamp','cmdb_id','message'])
else:
    error_window_display = pd.DataFrame(columns=['timestamp','cmdb_id','message'])

# Final compact outputs
log_result_df, error_window_display

```
Out[4]:
```
Summary (plain English):

Scope: I checked log.csv and error_logs.csv for cmdb_ids ["ts-auth-service","ts-order-service","ts-route-service","ts-travel-service"] inside 2024-01-23 16:03:20 → 16:33:20 UTC.

log.csv findings:
- Common facts
  - Each reported (cmdb_id, log_name) has 25 samples in the window.
  - log.error_count: global P95 = 0.0 for all four services; count_in_window = 25 and anomaly_count = 25. Earliest anomaly = 2024-01-23 16:06:00 UTC and max_value_in_window = 0.
    - Interpretation: P95 = 0.0 makes every zero-valued sample meet value >= P95, so these "anomalies" are a threshold artifact and not meaningful without understanding the metric semantics.
- log.row_count (more informative)
  - ts-auth-service
    - global_p95 = 340.0
    - count_in_window = 25, anomaly_count = 3
    - earliest_anomaly_timestamp = 2024-01-23 16:09:00 UTC
    - max_value_in_window = 344
    - sample window rows (first few): 2024-01-23T16:06:00+00:00 | 220, 2024-01-23T16:07:00+00:00 | 220, ...
  - ts-order-service
    - global_p95 = 454.6
    - count_in_window = 25, anomaly_count = 2
    - earliest_anomaly_timestamp = 2024-01-23 16:11:00 UTC
    - max_value_in_window = 515
    - sample window rows (first few): 2024-01-23T16:06:00+00:00 | 268, 2024-01-23T16:07:00+00:00 | 268, ...
  - ts-route-service
    - global_p95 = 1069.4
    - count_in_window = 25, anomaly_count = 2
    - earliest_anomaly_timestamp = 2024-01-23 16:11:00 UTC
    - max_value_in_window = 1164
    - sample window rows (first few): 2024-01-23T16:06:00+00:00 | 673, 2024-01-23T16:07:00+00:00 | 673, ...
  - ts-travel-service
    - global_p95 = 1970.6
    - count_in_window = 25, anomaly_count = 2
    - earliest_anomaly_timestamp = 2024-01-23 16:11:00 UTC
    - max_value_in_window = 2253
    - sample window rows (first few): 2024-01-23T16:06:00+00:00 | 1181, 2024-01-23T16:07:00+00:00 | 1181, ...

error_logs.csv findings:
- No rows in the incident window for these services (empty result).

Practical takeaways:
- log.error_count results are not reliable indicators here because global P95 = 0 makes all zeros flagged as "anomalies" — confirm metric semantics before acting on them.
- log.row_count shows real spike candidates: ts-auth-service (3 samples above P95), and ts-order, ts-route, ts-travel (2 samples each). The earliest notable spikes are around 16:09–16:11 UTC; peak row_count values reach ~344 (auth), ~515 (order), ~1,164 (route), and ~2,253 (travel).
- Next steps: investigate the timestamps listed for the row_count spikes and correlate with metrics/traces/logs (CPU, latency, mem, trace durations) around 16:09–16:29 UTC to determine cause.

The original code execution output of IPython Kernel is also provided below for reference:

(             cmdb_id         log_name  global_p95  count_in_window  anomaly_count earliest_anomaly_timestamp  max_value_in_window                        sample_window_rows_up_to_10
0    ts-auth-service  log.error_count         0.0               25             25  2024-01-23 16:06:00+00:00                    0  [2024-01-23T16:06:00+00:00 | 0, 2024-01-23T16:...
1   ts-order-service  log.error_count         0.0               25             25  2024-01-23 16:06:00+00:00                    0  [2024-01-23T16:06:00+00:00 | 0, 2024-01-23T16:...
2   ts-route-service  log.error_count         0.0               25             25  2024-01-23 16:06:00+00:00                    0  [2024-01-23T16:06:00+00:00 | 0, 2024-01-23T16:...
3  ts-travel-service  log.error_count         0.0               25             25  2024-01-23 16:06:00+00:00                    0  [2024-01-23T16:06:00+00:00 | 0, 2024-01-23T16:...
4    ts-auth-service    log.row_count       340.0               25              3  2024-01-23 16:09:00+00:00                  344  [2024-01-23T16:06:00+00:00 | 220, 2024-01-23T1...
5   ts-order-service    log.row_count       454.6               25              2  2024-01-23 16:11:00+00:00                  515  [2024-01-23T16:06:00+00:00 | 268, 2024-01-23T1...
6   ts-route-service    log.row_count      1069.4               25              2  2024-01-23 16:11:00+00:00                 1164  [2024-01-23T16:06:00+00:00 | 673, 2024-01-23T1...
7  ts-travel-service    log.row_count      1970.6               25              2  2024-01-23 16:11:00+00:00                 2253  [2024-01-23T16:06:00+00:00 | 1181, 2024-01-23T..., Empty DataFrame
Columns: [timestamp, cmdb_id, message]
Index: [])```
```

In [ ]:
# In[5]:

import pandas as pd

# reuse trace_df and incident window if present; otherwise load and parse
try:
    trace_df
except NameError:
    trace_df = pd.read_csv('trace.csv')
    trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)

# ensure timestamp is datetime UTC
if not pd.api.types.is_datetime64_any_dtype(trace_df['timestamp']):
    trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)

# parameters
substrs = ["ts-auth-service","ts-order-service","ts-route-service","ts-travel-service"]
start = pd.to_datetime('2024-01-23 16:03:20', utc=True)
end   = pd.to_datetime('2024-01-23 16:33:20', utc=True)

# find trace_name values that contain any of the substrings (match anywhere)
all_trace_names = pd.Series(trace_df['trace_name'].dropna().unique())
mask = all_trace_names.apply(lambda tn: any(s in tn for s in substrs))
matching_trace_names = all_trace_names[mask].tolist()

# compute global P95 per trace_name using full available series (across all cmdb_ids)
trace_p95 = (trace_df[trace_df['trace_name'].isin(matching_trace_names)]
             .dropna(subset=['value'])
             .groupby('trace_name', sort=False)['value']
             .quantile(0.95)
             .reset_index()
             .rename(columns={'value':'global_p95'}))

# filter rows to incident window
trace_window = trace_df[
    (trace_df['trace_name'].isin(matching_trace_names)) &
    (trace_df['timestamp'] >= start) &
    (trace_df['timestamp'] <= end)
].copy()

# prepare per-trace_name summaries
rows = []
if not trace_window.empty:
    win_with_p95 = trace_window.merge(trace_p95, on='trace_name', how='left')
    grp = win_with_p95.groupby('trace_name', sort=False)
    for tname, sub in grp:
        count_in_window = int(len(sub))
        gp95 = sub['global_p95'].iloc[0] if 'global_p95' in sub.columns else float('nan')
        if pd.notna(gp95):
            anomalies = sub[sub['value'] >= gp95].sort_values('timestamp', ascending=True)
        else:
            anomalies = sub.iloc[0:0]
        anomaly_count = int(len(anomalies))
        earliest_anom_ts = anomalies['timestamp'].min() if anomaly_count>0 else pd.NaT
        max_value = sub['value'].max() if count_in_window>0 else None
        sample_anoms = []
        if anomaly_count>0:
            samp = anomalies.head(5)
            sample_anoms = [f"{ts.isoformat()} | {val}" for ts, val in zip(samp['timestamp'], samp['value'])]
        rows.append({
            'trace_name': tname,
            'global_p95': gp95,
            'count_in_window': count_in_window,
            'anomaly_count': anomaly_count,
            'earliest_anomaly_timestamp': earliest_anom_ts,
            'max_value_in_window': max_value,
            'sample_anomalies_up_to_5': sample_anoms
        })

trace_name_summary = pd.DataFrame(rows)

# only include trace_name entries with count_in_window > 0, sort and limit to top 20
if not trace_name_summary.empty:
    trace_name_summary = (trace_name_summary[trace_name_summary['count_in_window']>0]
                          .sort_values(['anomaly_count','trace_name'], ascending=[False, True])
                          .reset_index(drop=True)
                          .head(20))

# final compact output
trace_name_summary

```
Out[5]:
```
Summary (plain English):

Scope: I searched trace_name values that mention any of ["ts-auth-service","ts-order-service","ts-route-service","ts-travel-service"] and analyzed them in the incident window 2024-01-23 16:03:20 → 16:33:20 UTC.

High-level note:
- Many trace error_rate series have global P95 = 0.0 and all window samples = 0.0, so anomaly_count = count_in_window. That is a threshold artifact (zeros flagged as >= P95) and not a meaningful elevated error signal.

Meaningful signals (duration / row_count spikes) — top findings:
1. trace.to_ts-order-service (across cmdb rows)
   - duration_mean: global_p95 ≈ 0.7635, count_in_window = 172, anomaly_count = 9, earliest anomaly 2024-01-23 16:20:00 UTC, max ≈ 0.8740
   - duration_p95: global_p95 ≈ 0.8190, count_in_window = 172, anomaly_count = 9, earliest anomaly 2024-01-23 16:11:00 UTC, max ≈ 1.1475
   - row_count: global_p95 = 463.25, count_in_window = 172, anomaly_count = 9, earliest anomaly 2024-01-23 16:09:00 UTC, max = 608

2. trace.from_ts-travel-service (across cmdb rows)
   - duration_mean: global_p95 ≈ 0.04178, count_in_window = 150, anomaly_count = 8, earliest anomaly 2024-01-23 16:06:00 UTC, max ≈ 0.04774
   - duration_p95: global_p95 ≈ 0.05589, count_in_window = 150, anomaly_count = 8, earliest anomaly 2024-01-23 16:16:00 UTC, max ≈ 0.07166
   - row_count: global_p95 = 1508.8, count_in_window = 150, anomaly_count = 8, earliest anomaly 2024-01-23 16:07:00 UTC, max = 2026

3. trace.to_ts-travel-service
   - duration_mean: global_p95 ≈ 0.5126, count_in_window = 148, anomaly_count = 8, earliest anomaly 2024-01-23 16:07:00 UTC, max ≈ 0.5992
   - duration_p95: global_p95 ≈ 0.7609, count_in_window = 148, anomaly_count = 8, earliest anomaly 2024-01-23 16:19:00 UTC, max ≈ 0.9798
   - row_count: global_p95 = 1510.4, count_in_window = 148, anomaly_count = 8, earliest anomaly 2024-01-23 16:07:00 UTC, max = 2026

4. trace.to_ts-route-service
   - duration_mean: global_p95 ≈ 0.003825, count_in_window = 100, anomaly_count = 5, earliest anomaly 2024-01-23 16:18:00 UTC, max ≈ 0.004961
   - duration_p95: global_p95 ≈ 0.004366, count_in_window = 100, anomaly_count = 5, earliest anomaly 2024-01-23 16:14:00 UTC, max ≈ 0.005205
   - row_count: global_p95 = 2076.5, count_in_window = 100, anomaly_count = 5, earliest anomaly 2024-01-23 16:09:00 UTC, max = 2328

Other notes:
- Several trace.error_rate names reference the services and show 100% "anomalies" but all values are 0.0 — treat these as non-actionable without understanding encoding.
- The duration/row_count anomalies concentrate on ts-order-service and ts-travel-service flows (both directions), with earliest spikes around ~16:07–16:11 and further spikes near ~16:19–16:20 for durations.

Recommended next steps:
- Ignore the error_rate P95=0 artifact unless you confirm zero means an error.  
- Investigate the trace durations and row_count spikes for:
  - to_ts-order-service (notable at ~16:09 and ~16:20),
  - from/to ts-travel-service (spikes starting ~16:07 and duration spike ~16:16–16:19),
  - to_ts-route-service (spikes ~16:09–16:18).
- Correlate these trace spikes with the metric/log signals previously observed (e.g., row_count and resource/latency anomalies) around 16:06–16:29 UTC to narrow root cause.

The original code execution output of IPython Kernel is also provided below for reference:

trace_name   global_p95  count_in_window  anomaly_count earliest_anomaly_timestamp  max_value_in_window                           sample_anomalies_up_to_5
0         trace.to_ts-order-service.error_rate     0.000000              172            172  2024-01-23 16:06:00+00:00             0.000000  [2024-01-23T16:06:00+00:00 | 0.0, 2024-01-23T1...
1      trace.from_ts-travel-service.error_rate     0.000000              150            150  2024-01-23 16:06:00+00:00             0.000000  [2024-01-23T16:06:00+00:00 | 0.0, 2024-01-23T1...
2        trace.to_ts-travel-service.error_rate     0.000000              148            148  2024-01-23 16:06:00+00:00             0.000000  [2024-01-23T16:06:00+00:00 | 0.0, 2024-01-23T1...
3         trace.to_ts-route-service.error_rate     0.000000              100            100  2024-01-23 16:06:00+00:00             0.000000  [2024-01-23T16:06:00+00:00 | 0.0, 2024-01-23T1...
4       trace.from_ts-order-service.error_rate     0.000000               50             50  2024-01-23 16:06:00+00:00             0.000000  [2024-01-23T16:06:00+00:00 | 0.0, 2024-01-23T1...
5          trace.to_ts-auth-service.error_rate     0.000000               50             50  2024-01-23 16:06:00+00:00             0.000000  [2024-01-23T16:06:00+00:00 | 0.0, 2024-01-23T1...
6        trace.from_ts-auth-service.error_rate     0.000000               25             25  2024-01-23 16:06:00+00:00             0.000000  [2024-01-23T16:06:00+00:00 | 0.0, 2024-01-23T1...
7       trace.from_ts-route-service.error_rate     0.000000               25             25  2024-01-23 16:06:00+00:00             0.000000  [2024-01-23T16:06:00+00:00 | 0.0, 2024-01-23T1...
8      trace.to_ts-order-service.duration_mean     0.763459              172              9  2024-01-23 16:20:00+00:00             0.874014  [2024-01-23T16:20:00+00:00 | 0.767225625, 2024...
9       trace.to_ts-order-service.duration_p95     0.818965              172              9  2024-01-23 16:11:00+00:00             1.147465  [2024-01-23T16:11:00+00:00 | 0.828446249999999...
10         trace.to_ts-order-service.row_count   463.250000              172              9  2024-01-23 16:09:00+00:00           608.000000  [2024-01-23T16:09:00+00:00 | 484.0, 2024-01-23...
11  trace.from_ts-travel-service.duration_mean     0.041781              150              8  2024-01-23 16:06:00+00:00             0.047740  [2024-01-23T16:06:00+00:00 | 0.043187306666666...
12   trace.from_ts-travel-service.duration_p95     0.055891              150              8  2024-01-23 16:16:00+00:00             0.071657  [2024-01-23T16:16:00+00:00 | 0.056395949999999...
13      trace.from_ts-travel-service.row_count  1508.800000              150              8  2024-01-23 16:07:00+00:00          2026.000000  [2024-01-23T16:07:00+00:00 | 1626.0, 2024-01-2...
14    trace.to_ts-travel-service.duration_mean     0.512583              148              8  2024-01-23 16:07:00+00:00             0.599189  [2024-01-23T16:07:00+00:00 | 0.544448826086956...
15     trace.to_ts-travel-service.duration_p95     0.760930              148              8  2024-01-23 16:19:00+00:00             0.979768  [2024-01-23T16:19:00+00:00 | 0.933806199999999...
16        trace.to_ts-travel-service.row_count  1510.400000              148              8  2024-01-23 16:07:00+00:00          2026.000000  [2024-01-23T16:07:00+00:00 | 1626.0, 2024-01-2...
17     trace.to_ts-route-service.duration_mean     0.003825              100              5  2024-01-23 16:18:00+00:00             0.004961  [2024-01-23T16:18:00+00:00 | 0.004097584615384...
18      trace.to_ts-route-service.duration_p95     0.004366              100              5  2024-01-23 16:14:00+00:00             0.005205  [2024-01-23T16:14:00+00:00 | 0.005205199999999...
19         trace.to_ts-route-service.row_count  2076.500000              100              5  2024-01-23 16:09:00+00:00          2328.000000  [2024-01-23T16:09:00+00:00 | 2132.0, 2024-01-2...```
```

In [ ]:
# In[6]:

import pandas as pd

# reuse trace_df if available; otherwise load and parse timestamps
try:
    trace_df
except NameError:
    trace_df = pd.read_csv('trace.csv')
    trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)

# ensure timestamp dtype is datetime UTC
if not pd.api.types.is_datetime64_any_dtype(trace_df['timestamp']):
    trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)

# target trace_name patterns (explicit full names)
target_trace_names = [
    "trace.to_ts-order-service.duration_mean",
    "trace.to_ts-order-service.duration_p95",
    "trace.to_ts-order-service.row_count",
    "trace.from_ts-travel-service.duration_mean",
    "trace.from_ts-travel-service.duration_p95",
    "trace.from_ts-travel-service.row_count",
    "trace.to_ts-travel-service.duration_mean",
    "trace.to_ts-travel-service.duration_p95",
    "trace.to_ts-travel-service.row_count",
    "trace.to_ts-route-service.duration_mean",
    "trace.to_ts-route-service.duration_p95",
    "trace.to_ts-route-service.row_count"
]

# Incident window (UTC)
start = pd.to_datetime('2024-01-23 16:03:20', utc=True)
end   = pd.to_datetime('2024-01-23 16:33:20', utc=True)

# 1) Compute global P95 per trace_name across the full series (thresholds before filtering)
available = trace_df[trace_df['trace_name'].isin(target_trace_names)].copy()
if not available.empty:
    trace_p95 = (
        available
        .dropna(subset=['value'])
        .groupby('trace_name', sort=False)['value']
        .quantile(0.95)
        .reset_index()
        .rename(columns={'value': 'global_p95'})
    )
else:
    trace_p95 = pd.DataFrame(columns=['trace_name','global_p95'])

# 2) Filter rows to the incident window for the listed trace_names
window_df = trace_df[
    (trace_df['trace_name'].isin(target_trace_names)) &
    (trace_df['timestamp'] >= start) &
    (trace_df['timestamp'] <= end)
].copy()

# 3) Join global_p95 into window rows and select rows where value >= global_p95
if window_df.empty:
    anomalies_head = pd.DataFrame(columns=['timestamp','cmdb_id','trace_name','value','global_p95'])
else:
    win_with_p95 = window_df.merge(trace_p95, on='trace_name', how='left')
    # keep only rows with a computed global_p95 and value >= global_p95
    cond = win_with_p95['global_p95'].notna() & (win_with_p95['value'] >= win_with_p95['global_p95'])
    anomalies = win_with_p95.loc[cond, ['timestamp','cmdb_id','trace_name','value','global_p95']].copy()
    # sort by timestamp ascending and keep earliest 20 matches
    anomalies_head = anomalies.sort_values('timestamp', ascending=True).head(20).reset_index(drop=True)

# display compact result (DataFrame)
anomalies_head

```
Out[6]:
```
Summary (plain English):

I inspected the requested trace metrics and returned the earliest 20 rows in the incident window where the trace value exceeded its global P95. Key findings:

- First anomalies appear at 2024-01-23 16:06–16:07 UTC:
  - 16:06 — trace.from_ts-travel-service.duration_mean (cmdb_id=ts-seat-service) = 0.04319 > p95 0.04178
  - 16:07 — trace.to_ts-travel-service.duration_mean (cmdb_id=root) = 0.54445 > p95 0.51258
  - 16:07 — high row_count for ts-travel-service (both from_ts-travel-service.row_count and to_ts-travel-service.row_count = 1626 > p95 ~1508 / ~1510)

- Repeated row_count spikes across services between ~16:07 and ~16:14 UTC:
  - ts-travel-service row_count above p95 at 16:07, 16:09, 16:11, 16:13, 16:14 (values up to 2026).
  - ts-travel-service to_ts-travel-service.row_count similarly high at same times (values up to 2026).
  - ts-order-service to_ts-order-service.row_count exceeded p95 at 16:09, 16:11, 16:13, 16:14 (examples: 484, 608, 537, 529; p95 = 463.25).
  - ts-route-service to_ts-route-service.row_count exceeded p95 at 16:09 and 16:11 (2132, 2328; p95 = 2076.5).

- Notable duration anomalies:
  - 16:11 — trace.to_ts-order-service.duration_p95 (cmdb_id=root) = 0.82845 > p95 0.81897 (duration_p95 spike).
  - Several duration_mean entries for travel/order flows exceeded their p95s (e.g., to_ts-travel-service at 16:07–16:08).

- Temporal pattern: concentrated cluster of spikes starts ~16:07 and recurs through ~16:14, with another duration-related spike noted earlier in prior analysis around 16:20. The largest magnitude spikes in this selection are row_count increases (order/route/travel flows) at ~16:09–16:11.

Interpretation and recommended next steps:
- The traces indicate increased traffic/row_count on flows involving ts-travel-service, ts-order-service and ts-route-service starting ~16:07 and peaking around ~16:09–16:11; some duration percentiles also exceeded normal at ~16:11 and ~16:20.
- Correlate these timestamps with service metrics (CPU, memory, diskio), logs, and traces for the same times to determine whether higher load caused increased latencies or downstream resource saturation.

The original code execution output of IPython Kernel is also provided below for reference:

timestamp            cmdb_id                                  trace_name        value   global_p95
0  2024-01-23 16:06:00+00:00    ts-seat-service  trace.from_ts-travel-service.duration_mean     0.043187     0.041781
1  2024-01-23 16:07:00+00:00               root    trace.to_ts-travel-service.duration_mean     0.544449     0.512583
2  2024-01-23 16:07:00+00:00  ts-travel-service      trace.from_ts-travel-service.row_count  1626.000000  1508.800000
3  2024-01-23 16:07:00+00:00  ts-travel-service        trace.to_ts-travel-service.row_count  1626.000000  1510.400000
4  2024-01-23 16:08:00+00:00               root    trace.to_ts-travel-service.duration_mean     0.517946     0.512583
5  2024-01-23 16:09:00+00:00   ts-order-service         trace.to_ts-order-service.row_count   484.000000   463.250000
6  2024-01-23 16:09:00+00:00   ts-route-service         trace.to_ts-route-service.row_count  2132.000000  2076.500000
7  2024-01-23 16:09:00+00:00  ts-travel-service      trace.from_ts-travel-service.row_count  1516.000000  1508.800000
8  2024-01-23 16:09:00+00:00  ts-travel-service        trace.to_ts-travel-service.row_count  1516.000000  1510.400000
9  2024-01-23 16:11:00+00:00  ts-travel-service        trace.to_ts-travel-service.row_count  2026.000000  1510.400000
10 2024-01-23 16:11:00+00:00  ts-travel-service      trace.from_ts-travel-service.row_count  2026.000000  1508.800000
11 2024-01-23 16:11:00+00:00   ts-order-service         trace.to_ts-order-service.row_count   608.000000   463.250000
12 2024-01-23 16:11:00+00:00               root      trace.to_ts-order-service.duration_p95     0.828446     0.818965
13 2024-01-23 16:11:00+00:00   ts-route-service         trace.to_ts-route-service.row_count  2328.000000  2076.500000
14 2024-01-23 16:13:00+00:00   ts-order-service         trace.to_ts-order-service.row_count   537.000000   463.250000
15 2024-01-23 16:13:00+00:00  ts-travel-service      trace.from_ts-travel-service.row_count  1698.000000  1508.800000
16 2024-01-23 16:13:00+00:00  ts-travel-service        trace.to_ts-travel-service.row_count  1698.000000  1510.400000
17 2024-01-23 16:14:00+00:00  ts-travel-service      trace.from_ts-travel-service.row_count  1751.000000  1508.800000
18 2024-01-23 16:14:00+00:00   ts-basic-service      trace.to_ts-route-service.duration_p95     0.005205     0.004366
19 2024-01-23 16:14:00+00:00   ts-order-service         trace.to_ts-order-service.row_count   529.000000   463.250000```
```